In [1]:
!pip install selenium webdriver-manager

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time
import os

def check_lazada_html(url, save_file="lazada_html.html"):
    chrome_options = Options()
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-infobars")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    print("Đang khởi động trình duyệt ChromeDriver...")
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    try:
        print(f"Đang truy cập: {url}")
        driver.get(url)
        
        print("Đang đợi trang load 10 giây (bạn hãy giải captcha nếu có)...")
        time.sleep(10) # Tăng thời gian chờ lên một xíu để bạn có thời gian kéo captcha nếu bị hỏi
        
        # Cuộn trang từ từ để nạp hết các sản phẩm và ảnh
        for i in range(5):
            driver.execute_script(f"window.scrollTo(0, {(i+1)*800});")
            time.sleep(1)
        
        html_source = driver.page_source
        
        # LƯU FILE HTML
        with open(save_file, "w", encoding="utf-8") as f:
            f.write(html_source)
            
        print("-" * 50)
        print(f"✅ THÀNH CÔNG: Đã lưu mã HTML vào file {os.path.abspath(save_file)}!")
        print(f"Độ dài mã HTML: {len(html_source)} ký tự.")
        print("-" * 50)
        
        return html_source

    except Exception as e:
        print(f"❌ Xảy ra lỗi: {e}")
        return None
        
    finally:
        driver.quit()
        print("Đã đóng trình duyệt.")


In [3]:
from bs4 import BeautifulSoup
import pandas as pd
import re

def extract_lazada_data(html_file_path, category):
    print(f"Đang đọc file HTML: {html_file_path}")
    
    # Mở và đọc file HTML (nhớ dùng utf-8 nếu không sẽ bị lỗi font tiếng Việt)
    with open(html_file_path, "r", encoding="utf-8") as file:
        html_content = file.read()
        
    # Tạo đối tượng BeautifulSoup để parse HTML
    soup = BeautifulSoup(html_content, "html.parser")
    
    # Tìm TẤT CẢ các thẻ div chứa thông tin của một sản phẩm
    # Lazada dùng thuộc tính data-qa-locator="product-item" để đánh dấu
    product_items = soup.find_all(attrs={"data-qa-locator": "product-item"})
    print(f"✅ Tìm thấy {len(product_items)} sản phẩm trong trang!")
    
    parsed_data = []
    
    for item in product_items:
        # 1. Tên, Link và ID
        a_tag = item.find("a", title=True)
        title = a_tag["title"] if a_tag else None
        
        link = a_tag["href"] if a_tag and "href" in a_tag.attrs else None
        product_id = None
        
        if link:
            if link.startswith("//"):
                link = "https:" + link
            # Dùng Regex để bắt một dãy số đứng sau chữ 'pdp-i' trong link
            id_match = re.search(r'pdp-i(\d+)\.html', link)
            if id_match:
                product_id = id_match.group(1)

            
        # 2. Giá sản phẩm
        price_tag = item.find("span", class_="ooOxS")
        price = price_tag.text.strip() if price_tag else None
        
        # 3. Giảm giá (VD: 50% Off)
        discount_tag = item.find("span", class_="IcOsH")
        discount = discount_tag.text.strip() if discount_tag else None
        
        # 4. Đã bán
        sold_tag = item.find("span", class_="_1cEkb")
        sold = sold_tag.text.strip() if sold_tag else None
        
        # 5. Số lượt đánh giá
        rating_tag = item.find("span", class_="qzqFw")
        rating_count = rating_tag.text.strip().replace("(", "").replace(")", "") if rating_tag else None
        
        
        rating_score = None

        # 6. Địa điểm bán (Hà Nội, Nước ngoài...)
        location_tag = item.find("span", class_="oa6ri")
        location = location_tag.text.strip() if location_tag else None
        
        # 7. Hình ảnh
        img_tag = item.find("img")
        image_url = img_tag["src"] if img_tag and "src" in img_tag.attrs else None

        # 8. Tên shop
        shop_name = None
        
        # 8. Shop URL
        shop_url = None
        
        parsed_data.append({
            "ID Sản phẩm": product_id,
            "Tên sản phẩm": title,
            "Danh mục": category,
            "Giá": price,
            "Giảm giá": discount,
            "Đã bán": sold,
            "Điểm đánh giá": rating_score, 
            "Lượt đánh giá": rating_count,
            "Địa điểm": location,
            "Đường dẫn": link,
            "Hình ảnh": image_url,
            "Tên shop": shop_name,
            "Shop URL": shop_url
        })
        
    return parsed_data




In [4]:
# ===================== CHẠY THỬ / CRAWL NHIỀU TRANG =====================
import os
import pandas as pd # Đảm bảo đã import pandas

# Danh sách các danh mục cần crawl cùng URL (đã chèn `{page}`) và số trang tương ứng
CATEGORIES = [
    {
        "name": "Điện thoại di động",
        "base_url": "https://www.lazada.vn/tag/%C4%90i%E1%BB%87n-tho%E1%BA%A1i-di-%C4%91%E1%BB%99ng/?page={page}&spm=a2o4n.searchlist.cate_1.1.13b9c553JRPJ6C&q=%C4%90i%E1%BB%87n%20tho%E1%BA%A1i%20di%20%C4%91%E1%BB%99ng&from=hp_categories&src=all_channel&catalog_redirect_tag=true",
        "total_pages": 1 
    },
    {
        "name": "Máy tính bảng",
        "base_url": "https://www.lazada.vn/catalog/?page={page}&spm=a2o4n.searchlist.cate_1.2.55cd4bebDvQjIi&q=M%C3%A1y%20t%C3%ADnh%20b%E1%BA%A3ng&from=hp_categories&src=all_channel",
        "total_pages": 1 # 
    },
    {
        "name": "Laptop",
        "base_url": "https://www.lazada.vn/catalog/?page={page}&spm=a2o4n.searchlist.cate_1.3.1c133753sGmzdp&q=Laptop&from=hp_categories&src=all_channel",
        "total_pages": 1 
    }
]

# --- THÊM PHẦN TẠO THƯ MỤC GỐC LƯU HTML ---
HTML_DIR = "lazada_tbdtu_html_pages" # Đổi tên thành thư mục thiết bị điện tử
if not os.path.exists(HTML_DIR):
    os.makedirs(HTML_DIR)
# -----------------------------

all_data = []

# Duyệt qua từng danh mục 
for category in CATEGORIES:
    cat_name = category["name"]
    base_url = category["base_url"]
    total_pages = category["total_pages"]
    
    print("\n" + "=" * 50)
    print(f"BẮT ĐẦU CRAWL DANH MỤC: {cat_name.upper()}")
    print("=" * 50)
    
    # Tạo thư mục con cho mỗi danh mục bên trong HTML_DIR (tuỳ chọn cho gọn)
    cat_html_dir = os.path.join(HTML_DIR, cat_name)
    if not os.path.exists(cat_html_dir):
        os.makedirs(cat_html_dir)
        
    for page in range(1, total_pages + 1):
        # Format lại url để gắn số trang vào
        url = base_url.format(page=page)
        
        # --- CẬP NHẬT ĐƯỜNG DẪN LƯU FILE VÀO THƯ MỤC ---
        html_file = os.path.join(cat_html_dir, f"lazada_{cat_name}_page_{page}.html")
        # ----------------------------------------------
        
        print(f"\n-[{cat_name}] [{page}/{total_pages}] Đang crawl trang {page}...")
        html_data = check_lazada_html(url, html_file)
        
        if html_data:
            if "product-item" in html_data or "Bm3ON" in html_data:
                print("🟢 CÓ dữ liệu sản phẩm!")
            else:
                print("🔴 KHÔNG tìm thấy sản phẩm — có thể bị chặn hoặc hết trang.")
            
            # Lưu ý truyền cat_name vào extract_lazada_data làm danh mục sản phẩm
            data = extract_lazada_data(html_file, cat_name)
            all_data.extend(data)
            print(f"→ Trang này: {len(data)} sản phẩm | Tổng cộng đang có: {len(all_data)}")
        else:
            print(f"Không lấy được HTML trang {page}, bỏ qua.")

# Tổng hợp và hiển thị sau khi đã crawl toàn bộ các danh mục
print("\n" + "=" * 50)
print("BƯỚC 3: TỔNG HỢP")
print("=" * 50)

df_raw = pd.DataFrame(all_data)
print(f"Tổng trước dedup : {len(df_raw)}")
print(f"ID bị null       : {df_raw['ID Sản phẩm'].isna().sum()}")

dupes = df_raw[df_raw.duplicated(subset=["ID Sản phẩm"], keep=False)]
print(f"Số dòng trùng ID : {len(dupes)}")

df = df_raw.drop_duplicates(subset=["ID Sản phẩm"], keep="first").reset_index(drop=True)
print(f"\nHOÀN THÀNH! Tổng sản phẩm duy nhất: {len(df)}")

display(df.head())

DATA_DIR = "data"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

# Nối đường dẫn lưu file
csv_path = os.path.join(DATA_DIR, "lazada_thietbidientu.csv")
# -------------------------------------

# Lưu file (utf-8-sig giúp mở bằng Excel không bị lỗi font Tiếng Việt)
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu thành công file CSV tổng tại: {csv_path}!")



BẮT ĐẦU CRAWL DANH MỤC: ĐIỆN THOẠI DI ĐỘNG

-[Điện thoại di động] [1/1] Đang crawl trang 1...
Đang khởi động trình duyệt ChromeDriver...
Đang truy cập: https://www.lazada.vn/tag/%C4%90i%E1%BB%87n-tho%E1%BA%A1i-di-%C4%91%E1%BB%99ng/?page=1&spm=a2o4n.searchlist.cate_1.1.13b9c553JRPJ6C&q=%C4%90i%E1%BB%87n%20tho%E1%BA%A1i%20di%20%C4%91%E1%BB%99ng&from=hp_categories&src=all_channel&catalog_redirect_tag=true
Đang đợi trang load 10 giây (bạn hãy giải captcha nếu có)...
--------------------------------------------------
✅ THÀNH CÔNG: Đã lưu mã HTML vào file c:\Users\ADMIN\Documents\Chuyen Nganh\HK2\Trực quan hóa dữ liệu\Lab 1\lazada_tbdtu_html_pages\Điện thoại di động\lazada_Điện thoại di động_page_1.html!
Độ dài mã HTML: 2936771 ký tự.
--------------------------------------------------
Đã đóng trình duyệt.
🟢 CÓ dữ liệu sản phẩm!
Đang đọc file HTML: lazada_tbdtu_html_pages\Điện thoại di động\lazada_Điện thoại di động_page_1.html
✅ Tìm thấy 40 sản phẩm trong trang!
→ Trang này: 40 sản phẩm | T

,ID Sản phẩm,Tên sản phẩm,Danh mục,Giá,Giảm giá,Đã bán,Điểm đánh giá,Lượt đánh giá,Địa điểm,Đường dẫn,Hình ảnh,Tên shop,Shop URL
0,3082255791,Điện thoại giá rẻ Samsung Galaxy A71 2sim ram ...,Điện thoại di động,298.000 ₫,33% Off,59 Đã bán,None,41,Hồ Chí Minh,https://www.lazada.vn/products/pdp-i3082255791...,https://img.lazcdn.com/g/p/e1ffde343f03629eb9c...,None,None
1,3305678831,"Điện thoại di động chắc chắn Oukitel WP32 Pro,...",Điện thoại di động,840.000 ₫,50% Off,None,None,None,Viet Nam,https://www.lazada.vn/products/pdp-i3305678831...,https://img.lazcdn.com/g/p/aa25539911b014152c9...,None,None
2,3065998302,Điện Thoại Thông Minh Giá Rẻ Samsung Galaxy J2...,Điện thoại di động,280.000 ₫,44% Off,95 Đã bán,None,22,Hà Nội,https://www.lazada.vn/products/pdp-i3065998302...,https://img.lazcdn.com/g/p/93aebb4ab6bb1f147ed...,None,None
3,2890686126,Điện thoại 5G giá rẻ Samsung Galaxy S10 5G 2si...,Điện thoại di động,299.000 ₫,27% Off,66 Đã bán,None,39,Hồ Chí Minh,https://www.lazada.vn/products/pdp-i2890686126...,https://img.lazcdn.com/g/p/104ff8fe1d5d91043de...,None,None
4,13312739208,Điện thoại di động M14 Ultra Chính Hãng đt giá...,Điện thoại di động,825.101 ₫,64% Off,6 Đã bán,None,3,Hà Nội,https://www.lazada.vn/products/pdp-i1331273920...,https://img.lazcdn.com/g/p/d612cba39cfd04a47ea...,None,None


Đã lưu thành công file CSV tổng tại: data\lazada_thietbidientu.csv!


In [5]:
import time
import re 
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from concurrent.futures import ThreadPoolExecutor
import threading
import queue
import pandas as pd
import random
import winsound  # 🔊 Thêm thư viện phát âm thanh trên Windows
import os

MAX_WORKERS = 3  # Số Chrome chạy song song

# ===================== SỬA LỖI ĐA LUỒNG =====================
# 1. Hàng đợi để phân phát ID Profile cho các luồng (Để chia làm 3 thư mục profile khác nhau)
profile_queue = queue.Queue()
for i in range(MAX_WORKERS):
    profile_queue.put(i)  # Thêm ID: 0, 1, 2 vào hàng đợi

# 2. Khóa (Lock) để tránh các luồng đụng độ nhau khi tạo file chromedriver
chrome_init_lock = threading.Lock()

def crawl_lazada_product(url):
    # Lấy 1 ID từ hàng đợi rảnh rỗi (Ngăn không cho luồng khác xài)
    profile_id = profile_queue.get()
    extracted_data = {}
    driver = None
    
    try:
        print(f"[Luồng {profile_id}] 🔄 Đang khởi tạo trình duyệt Chrome...")
        options = uc.ChromeOptions()
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        
        # CHIẾN THUẬT QUAN TRỌNG: MỖI LUỒNG DÙNG 1 FOLDER PROFILE KHÁC NHAU
        profile_path = rf'C:\Users\ADMIN\ChromeProfileData_{profile_id}'
        options.add_argument(f'--user-data-dir={profile_path}')
        
        #DÙNG KHÓA KHI KHỞI TẠO ĐỂ TRÁNH LỖI WINERROR 183
        with chrome_init_lock:
            driver = uc.Chrome(options=options)
            
        print(f"[Luồng {profile_id}] đang truy cập URL: {url}")
        driver.get(url)
        
        # --- CƠ CHẾ KIỂM TRA & VƯỢT BOT DETECTION ---
        time.sleep(3)
        if "we need to check if you are a robot" in driver.page_source.lower() or "recaptcha" in driver.page_source.lower() or "bảo vệ bởi ddos" in driver.page_source.lower():
            print(f"[Luồng {profile_id}] ⚠️ PHÁT HIỆN BOT! Vui lòng tự giải Captcha trên cửa sổ {profile_id}...")
            
            # ========================================================
            # Phát âm thanh cảnh báo bíp bíp liên tục (5 tiếng)
            try:
                for _ in range(5):
                    # winound.Beep(tần số_Hz, thời_gian_kêu_ms)
                    winsound.Beep(1500, 500) 
                    time.sleep(0.1)
            except:
                pass
            # ========================================================

            try:
                WebDriverWait(driver, 60).until(
                    EC.presence_of_element_located((By.CLASS_NAME, 'pdp-mod-review'))
                )
                print(f"[Luồng {profile_id}] ✅ Đã qua bước xác minh.")
            except Exception:
                print(f"[Luồng {profile_id}] ❌ Quá 60s không giải được.")
        
        # Chờ load trang
        try:
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CLASS_NAME, 'pdp-mod-review')))
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(4)
            driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(2)
        except Exception:
            pass
        
        # Trích xuất HTML
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        
        # 1. TÊN SHOP VÀ LINK SHOP
        seller_element = soup.select_one('.seller-name-v2__detail-name, .seller-name-v2__top a, .seller-name__detail')
        if seller_element:
            extracted_data['Shop Name'] = seller_element.get_text(strip=True)
            link = seller_element.get('href')
            if not link and seller_element.find_parent('a'):
                link = seller_element.find_parent('a').get('href')
            extracted_data['Shop URL'] = "https:" + link if link and link.startswith('//') else (link or "")
        else:
            extracted_data['Shop Name'] = ""
            extracted_data['Shop URL'] = ""
            
        # 2. ĐÁNH GIÁ TRUNG BÌNH
        score_tag = soup.select_one('.container-star-v2-score, .score-average, .score, .rating-score, .pdp-review-summary__stars')
        if score_tag:
            extracted_data['Rating Score'] = score_tag.get_text(strip=True)
        else:
            all_text = ' '.join(soup.stripped_strings)
            score_regex = re.search(r'(\d\.\d)\s*\(\d+\)', all_text)
            extracted_data['Rating Score'] = score_regex.group(1) if score_regex else "" # Để trống
            
    except Exception as e:
        print(f"[Luồng {profile_id}] ❌ Lỗi khi thao tác với Chrome: {e}")
    finally:
        # Quan trọng 1: Phải luôn tắt driver kể cả khi có lỗi
        try:
            if driver:
                driver.quit()
        except:
            pass
        # Quan trọng 2: Hoàn trả lại quyền dùng profile_id vào hàng đợi cho luồng sau dùng tiếp
        profile_queue.put(profile_id)
        
    return extracted_data


# ===================== QUẢN LÝ CHẠY ĐA LUỒNG =====================
# (Giả định bạn đã có biến df chứa số liệu ở các Cell trên)
df_missing = df[df['Điểm đánh giá'].isna() | df['Shop URL'].isna()].copy()
print(f"→ Sẽ crawl {len(df_missing)} sản phẩm với {MAX_WORKERS} luồng song song...")

def crawl_one(args):
    idx, row = args
    url = row["Đường dẫn"]
    
    if pd.isna(url):
        return idx, None
    
    # Random delay này vẫn giữ để giảm tải khi mới vào, kết hợp với "Lock" ở trên là an toàn tuyệt đối 100%
    time.sleep(random.uniform(1, 4)) 
    
    detail = crawl_lazada_product(url)
    if detail:
        print(f"✅ [{idx}] {row['Tên sản phẩm'][:30]} | Rating: {detail.get('Rating Score') or 'Rỗng'} | Shop: {detail.get('Shop Name') or 'Rỗng'}")
    return idx, detail

# Chạy đa luồng
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = executor.map(crawl_one, df_missing.iterrows())
    
    for idx, detail in futures:
        if detail is None or not detail:
            continue
        # Dùng 'in detail' để lưu cả những thành phần "" vào file excel cuối
        if "Rating Score" in detail:
            df.at[idx, "Điểm đánh giá"] = detail["Rating Score"]
        if "Shop Name" in detail:
            df.at[idx, "Tên shop"] = detail["Shop Name"]
        if "Shop URL" in detail:
            df.at[idx, "Shop URL"] = detail["Shop URL"]

# Đường dẫn lưu file
csv_path = os.path.join("data", "lazada_thietbidientu.csv")
# Lưu file (Đảm bảo tên file bạn muốn)
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"\n{'='*50}")
print(f"HOÀN THÀNH CÀO DỮ LIỆU!")
print(f"Rating đã có : {df['Điểm đánh giá'].notna().sum()}/{len(df)}")
print(f"Shop URL đã có: {df['Shop URL'].notna().sum()}/{len(df)}")
print(f"{'='*50}")
display(df.head())


→ Sẽ crawl 120 sản phẩm với 3 luồng song song...
[Luồng 0] 🔄 Đang khởi tạo trình duyệt Chrome...
[Luồng 1] 🔄 Đang khởi tạo trình duyệt Chrome...
[Luồng 2] 🔄 Đang khởi tạo trình duyệt Chrome...
[Luồng 0] đang truy cập URL: https://www.lazada.vn/products/pdp-i3065998302.html
[Luồng 1] đang truy cập URL: https://www.lazada.vn/products/pdp-i3305678831.html
[Luồng 2] đang truy cập URL: https://www.lazada.vn/products/pdp-i3082255791.html
✅ [2] Điện Thoại Thông Minh Giá Rẻ S | Rating: 4.6 | Shop: Smartchoice 247
[Luồng 0] 🔄 Đang khởi tạo trình duyệt Chrome...
✅ [0] Điện thoại giá rẻ Samsung Gala | Rating: 4.9 | Shop: BSNT.PN
[Luồng 0] đang truy cập URL: https://www.lazada.vn/products/pdp-i2890686126.html
[Luồng 2] 🔄 Đang khởi tạo trình duyệt Chrome...
[Luồng 2] đang truy cập URL: https://www.lazada.vn/products/pdp-i13312739208.html
✅ [1] Điện thoại di động chắc chắn O | Rating: 4.9 | Shop: Mỹ Duyênn Store
[Luồng 1] 🔄 Đang khởi tạo trình duyệt Chrome...
[Luồng 1] đang truy cập URL: https://www

,ID Sản phẩm,Tên sản phẩm,Danh mục,Giá,Giảm giá,Đã bán,Điểm đánh giá,Lượt đánh giá,Địa điểm,Đường dẫn,Hình ảnh,Tên shop,Shop URL
0,3082255791,Điện thoại giá rẻ Samsung Galaxy A71 2sim ram ...,Điện thoại di động,298.000 ₫,33% Off,59 Đã bán,4.9,41,Hồ Chí Minh,https://www.lazada.vn/products/pdp-i3082255791...,https://img.lazcdn.com/g/p/e1ffde343f03629eb9c...,BSNT.PN,https://www.lazada.vn/shop/bsnt-pn/?itemId=308...
1,3305678831,"Điện thoại di động chắc chắn Oukitel WP32 Pro,...",Điện thoại di động,840.000 ₫,50% Off,None,4.9,None,Viet Nam,https://www.lazada.vn/products/pdp-i3305678831...,https://img.lazcdn.com/g/p/aa25539911b014152c9...,Mỹ Duyênn Store,https://www.lazada.vn/shop/my-duyenn-store/?it...
2,3065998302,Điện Thoại Thông Minh Giá Rẻ Samsung Galaxy J2...,Điện thoại di động,280.000 ₫,44% Off,95 Đã bán,4.6,22,Hà Nội,https://www.lazada.vn/products/pdp-i3065998302...,https://img.lazcdn.com/g/p/93aebb4ab6bb1f147ed...,Smartchoice 247,https://www.lazada.vn/shop/smartchoice-247/?it...
3,2890686126,Điện thoại 5G giá rẻ Samsung Galaxy S10 5G 2si...,Điện thoại di động,299.000 ₫,27% Off,66 Đã bán,3.8,39,Hồ Chí Minh,https://www.lazada.vn/products/pdp-i2890686126...,https://img.lazcdn.com/g/p/104ff8fe1d5d91043de...,LeNgan _Store,https://www.lazada.vn/shop/lengan-store/?itemI...
4,13312739208,Điện thoại di động M14 Ultra Chính Hãng đt giá...,Điện thoại di động,825.101 ₫,64% Off,6 Đã bán,5,3,Hà Nội,https://www.lazada.vn/products/pdp-i1331273920...,https://img.lazcdn.com/g/p/d612cba39cfd04a47ea...,Pzi5i125,https://www.lazada.vn/shop/pzi5i125/?itemId=13...


In [ ]:
import time
import re
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from concurrent.futures import ThreadPoolExecutor
import threading
import queue
import pandas as pd
import random
import winsound
import os

MAX_WORKERS = 3  # Số luồng (cửa sổ Chrome) chạy song song

# ===================== KHỞI TẠO CƠ CHẾ ĐA LUỒNG =====================
# 1. Hàng đợi để phân phát ID Profile cho các luồng (0, 1, 2)
profile_queue = queue.Queue()
for i in range(MAX_WORKERS):
    profile_queue.put(i)

# 2. Khóa (Lock) để tránh các luồng đụng độ nhau lúc khởi tạo driver
chrome_init_lock = threading.Lock()

def crawl_and_parse_shop_profile(url):
    # Lấy 1 ID từ hàng đợi rảnh rỗi
    profile_id = profile_queue.get()
    
    # Template chứa dữ liệu mặc định (để nếu lỗi vẫn không bị thiếu cột)
    extracted_data = {
        'Shop URL': url,
        'Seller ID': None,
        'Loại cửa hàng': None,
        'Followers': None,
        'Đánh giá người bán': None,
        'Danh mục chính': None,
        'Đã tham gia': None,
        'Giao đúng hẹn': None,
        'Tỷ lệ hủy đơn': None,
        'Tỷ lệ phản hồi chat': None
    }
    
    driver = None
    html = "" # Biến chứa HTML lấy được
    
    try:
        print(f"[Luồng {profile_id}] Đang khởi tạo trình duyệt Chrome...")
        options = uc.ChromeOptions()
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        
        # 🟢 CHIẾN THUẬT QUAN TRỌNG CHỐNG "BÓP" HIỆU NĂNG KHI CHẠY ĐA LUỒNG
        # Ép Chrome vẫn phải tải Javascript ở mức tối đa dù cửa sổ bị ẩn/che khuất
        options.add_argument("--disable-background-timer-throttling")
        options.add_argument("--disable-backgrounding-occluded-windows")
        options.add_argument("--disable-renderer-backgrounding")
        
        # Mỗi luồng sử dụng một thư mục user-data riêng biệt
        profile_path = rf'C:\Users\ADMIN\ChromeProfileData_{profile_id}'
        options.add_argument(f'--user-data-dir={profile_path}')
        
        # Có Lock để tránh bị lỗi WinError 183
        with chrome_init_lock:
            driver = uc.Chrome(options=options)
            
        print(f"[Luồng {profile_id}] Đang truy cập URL Hồ Sơ Cửa Hàng: \n{url}")
        driver.get(url)
        
        # ==========================================================
        # --- CƠ CHẾ KIỂM TRA & VƯỢT BOT DETECTION ---
        time.sleep(3)
        page_source_lower = driver.page_source.lower()
        if "we need to check if you are a robot" in page_source_lower or "recaptcha" in page_source_lower or "bảo vệ bởi ddos" in page_source_lower:
            print(f"[Luồng {profile_id}] PHÁT HIỆN BOT! Vui lòng tự giải Captcha trên cửa sổ {profile_id}...")
            
            try:
                for _ in range(5):
                    winsound.Beep(1500, 500)
                    time.sleep(0.1)
            except:
                pass

            try:
                WebDriverWait(driver, 60).until(
                    EC.presence_of_element_located((By.CLASS_NAME, 'seller-name__detail')) 
                )
                print(f"[Luồng {profile_id}] Đã qua bước xác minh.")
            except Exception:
                print(f"[Luồng {profile_id}] Quá 60s không giải được vòng lặp.")
        # ==========================================================
        
        print(f"[Luồng {profile_id}] Đang đợi toàn bộ chỉ số uy tín tải về màn hình...")
        
        # Đợi cơ bản 10 giây để nạp khung hình
        time.sleep(10) 
        driver.execute_script("window.scrollBy(0, 500);")
        time.sleep(5)
        
        # 🟢 KIỂM TRA LINH HOẠT XEM THÔNG TIN ĐÃ LOAD LÊN DOM CHƯA
        # Cho phép kịch bản tối đa 3 lần đợi thêm. Nếu CPU lag khiến mạng rớt nhịp, máy sẽ thông minh xin chờ thêm 5 giây.
        for _ in range(3):
            temp_source = driver.page_source
            # Kiểm tra xem có 1 trong các chữ số thống kê được render thành công chưa
            if ("Giao đúng hẹn" in temp_source) or ("Shipped on time" in temp_source) or ("Đánh giá người bán" in temp_source) or ("Đã tham gia" in temp_source):
                break
            else:
                print(f"[Luồng {profile_id}] ⏳ Web bị lag chưa kịp đổ dữ liệu xuống, đang cố đợi thêm 5s...")
                time.sleep(5)
                driver.execute_script("window.scrollBy(0, 300);")
        
        # Chỉ lưu biến HTML trực tiếp lên RAM
        html = driver.page_source
        
    except Exception as e:
        print(f"[Luồng {profile_id}] Lỗi trong quá trình thao tác với Chrome: {e}")
    finally:
        # Luôn phải tắt driver để giải phóng RAM + trả profile lại hàng đợi
        try:
            if driver:
                driver.quit()
        except:
            pass
        profile_queue.put(profile_id)
        
    # --- PHÂN TÍCH DATA NGAY TỨC THÌ NẾU LẤY ĐƯỢC HTML ---
    if html:
        print(f"[Luồng {profile_id}] Đang bóc tách dữ liệu từ HTML...")
        soup = BeautifulSoup(html, 'html.parser')
        all_text = ' '.join(soup.stripped_strings)

        is_mall = bool(soup.find('img', alt='storeIcon'))
        extracted_data['Loại cửa hàng'] = 'LazMall' if is_mall else 'Shop Thường'

        seller_id_match = re.search(r'"sellerId"\s*:\s*(\d+)', html)
        if seller_id_match:
            extracted_data['Seller ID'] = seller_id_match.group(1)
        else: 
            shop_id_match = re.search(r'"shopId"\s*:\s*(\d+)', html)
            if shop_id_match:
                extracted_data['Seller ID'] = shop_id_match.group(1)

        followers = re.search(r'([0-9,]+)\s+Followers', all_text)
        if followers: extracted_data['Followers'] = followers.group(1)

        positive_rating = re.search(r'([0-9,.]+%)\s+(?:Đánh giá người bán|Seller Ratings)', all_text)
        if positive_rating: extracted_data['Đánh giá người bán'] = positive_rating.group(1)

        category_match = re.search(r'(?:Danh mục chính|Main category)(.*?)(?:Đã tham gia|Joined)', all_text)
        if category_match: 
            clean_category = re.sub(r'[^\w\s\'\-\&]', '', category_match.group(1)).strip()
            extracted_data['Danh mục chính'] = clean_category

        joined = re.search(r'(?:Đã tham gia|Joined)(?:.*?)\s+(\d+\s*\+?\s*(?:năm|years?|tháng|months?))', all_text)
        if joined: extracted_data['Đã tham gia'] = joined.group(1).strip()

        ship_on_time = re.search(r'(?:Giao đúng hẹn|Shipped on time)(?:.*?)\s+([0-9,.]+%|Không đủ dữ liệu|Not enough data)', all_text)
        if ship_on_time: extracted_data['Giao đúng hẹn'] = ship_on_time.group(1).strip()

        cancel_rate = re.search(r'(?:Tỷ lệ hủy đơn|Cancellation rate)(?:.*?)\s+([0-9,.]+%|Không đủ dữ liệu|Not enough data)', all_text)
        if cancel_rate: extracted_data['Tỷ lệ hủy đơn'] = cancel_rate.group(1).strip()

        chat_response = re.search(r'(?:tin nhắn trong 10|10 Minutes Response Rate)(?:.*?)\s+([0-9,.]+%)', all_text)
        if chat_response: extracted_data['Tỷ lệ phản hồi chat'] = chat_response.group(1).strip()
        
        print(f"[Luồng {profile_id}] ✅ Đã trích xuất xong dữ liệu cho một shop.")
        
    return extracted_data

raw_urls = df['Shop URL'].dropna().unique()

danh_sach_shop_urls = []
for url in raw_urls:
    # Link cũ thường có dạng: https://www.lazada.vn/shop/abc-store/?itemId=123...
    # split('?') sẽ cắt lấy phần gốc: https://www.lazada.vn/shop/abc-store/
    base_url = url.split('?')[0] 
    
    # Ghép thêm đuôi đi thẳng vào Tab Profile của shop đó
    profile_url = f"{base_url}?path=profile.htm&pageTypeId=3"
    
    # Tránh trường hợp 1 shop có nhiều link kiểu base_url giống nhau nhưng parameter # nhau mọc ra
    if profile_url not in danh_sach_shop_urls:
        danh_sach_shop_urls.append(profile_url)

print(f"Tổng số Shop cần cào thông tin hồ sơ (đã lọc trùng): {len(danh_sach_shop_urls)}")


# ===================== THỰC THI CHẠY ĐA LUỒNG =====================

def wrapper_process(url):
    time.sleep(random.uniform(1, 4)) # Delay nhẹ cho đỡ gửi dồn dập
    return crawl_and_parse_shop_profile(url)

print(f"→ Sẽ crawl {len(danh_sach_shop_urls)} Shop URL với {MAX_WORKERS} luồng song song...\n")

all_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = executor.map(wrapper_process, danh_sach_shop_urls)
    
    for result in futures:
        if result:
            all_results.append(result)

# ===== TỔNG HỢP VÀ GHI RA FILE CSV =====
df_profile = pd.DataFrame(all_results)
# Đổi tên biến sang csv_file và để đuôi .csv
csv_file = os.path.join("data", "lazada_shop_profiles_tbdtu.csv")
# Sử dụng hàm df_profile.to_csv thay vì to_excel
# Thêm utf-8-sig để đọc tiếng Việt chuẩn xác trong Excel
df_profile.to_csv(csv_file, index=False, encoding="utf-8-sig")
print(f"Đã lưu thành công file CSV tại: {csv_file}!")

print(f"\n====== HOÀN TẤT ======")
print(f"✅ Đã xử lý tổng cộng {len(all_results)} shop.")

# Hiển thị xem trước
display(df_profile.head())




Tổng số Shop cần cào thông tin hồ sơ (đã lọc trùng): 60
→ Sẽ crawl 60 Shop URL với 3 luồng song song...

[Luồng 0] Đang khởi tạo trình duyệt Chrome...
[Luồng 1] Đang khởi tạo trình duyệt Chrome...
[Luồng 2] Đang khởi tạo trình duyệt Chrome...
[Luồng 0] Đang truy cập URL Hồ Sơ Cửa Hàng: 
https://www.lazada.vn/shop/smartchoice-247/?path=profile.htm&pageTypeId=3
[Luồng 1] Đang truy cập URL Hồ Sơ Cửa Hàng: 
https://www.lazada.vn/shop/my-duyenn-store/?path=profile.htm&pageTypeId=3
[Luồng 2] Đang truy cập URL Hồ Sơ Cửa Hàng: 
https://www.lazada.vn/shop/bsnt-pn/?path=profile.htm&pageTypeId=3
[Luồng 0] Đang đợi toàn bộ chỉ số uy tín tải về màn hình...
[Luồng 1] Đang đợi toàn bộ chỉ số uy tín tải về màn hình...
[Luồng 2] Đang đợi toàn bộ chỉ số uy tín tải về màn hình...
[Luồng 0] Đang bóc tách dữ liệu từ HTML...
[Luồng 0] ✅ Đã trích xuất xong dữ liệu cho một shop.
[Luồng 1] Đang bóc tách dữ liệu từ HTML...
[Luồng 1] ✅ Đã trích xuất xong dữ liệu cho một shop.
[Luồng 2] Đang bóc tách dữ liệu từ H

NameError: name 'excel_file' is not defined

Đã lưu thành công file CSV tại: data\lazada_shop_profiles_tbdtu.csv!

====== HOÀN TẤT ======
✅ Đã xử lý tổng cộng 60 shop.


,Shop URL,Seller ID,Loại cửa hàng,Followers,Đánh giá người bán,Danh mục chính,Đã tham gia,Giao đúng hẹn,Tỷ lệ hủy đơn,Tỷ lệ phản hồi chat
0,https://www.lazada.vn/shop/bsnt-pn/?path=profi...,200805088772,Shop Thường,1078,96%,Mobiles & Tablets,1 + năm,62%,None,37.14%
1,https://www.lazada.vn/shop/my-duyenn-store/?pa...,201088736139,Shop Thường,2,None,Mobiles & Tablets,None,None,None,100.00%
2,https://www.lazada.vn/shop/smartchoice-247/?pa...,200169657278,Shop Thường,3775,83%,Mobiles & Tablets,4 + năm,None,None,66.67%
3,https://www.lazada.vn/shop/lengan-store/?path=...,200784512989,Shop Thường,1525,95%,Mobiles & Tablets,1 + năm,51%,None,37.84%
4,https://www.lazada.vn/shop/pzi5i125/?path=prof...,200551472302,Shop Thường,174,67%,Mobiles & Tablets,2 + năm,95%,None,94.12%
